In [4]:
import pandas as pd
from pathlib import Path
import sys
import numpy as np
processed_dir = Path(r"E:\Coding Site\just_for_fun\Electoral_Modelling\csv_output")

for csv_path in processed_dir.glob("*.csv"):
    globals()[csv_path.stem] = pd.read_csv(csv_path)

sub_region_map = pd.read_csv(r"E:\Coding Site\just_for_fun\Info_CSV\sub_region_map.csv")
df_electorate_demographics = pd.read_csv(r"E:\Coding Site\just_for_fun\Electoral_Modelling\csv_output\Electorate_Demographics.csv")
df_turnout_spoil = pd.read_csv(r"E:\Coding Site\just_for_fun\Info_CSV\Electoral_Modelling\Turnout_Spoil_Data.csv")
df_result_first_round = pd.read_csv(r"E:\Coding Site\just_for_fun\Info_CSV\Electoral_Modelling\Election_Result_first_round.csv")

In [213]:
polygon_cols_to_drop = [c for c in ["ward_immigrant", "ward_localists"] if c in df_polygon.columns]
if polygon_cols_to_drop:
    df_polygon = df_polygon.drop(columns=polygon_cols_to_drop)

ward_lookup = (
    df_districts[["District", "District_Ward", "ward_immigrant", "ward_localists"]]
    .drop_duplicates(subset=["District", "District_Ward"])
)

df_polygon = df_polygon.merge(ward_lookup, on=["District", "District_Ward"], how="left")

cluster_cols = ["S", "A1", "A2", "B1", "B2", "C1", "C2", "D1", "D2", "E1", "E2", "F1", "F2"]

polygon_cols_to_drop = [c for c in cluster_cols if c in df_polygon.columns]
if polygon_cols_to_drop:
    df_polygon = df_polygon.drop(columns=polygon_cols_to_drop)

community_lookup = (
    community_df[["Region", "Type", "Code"] + cluster_cols]
    .drop_duplicates(subset=["Region", "Type", "Code"])
)

df_polygon = df_polygon.merge(
    community_lookup,
    left_on=["Region", "Location", "County_Type"],
    right_on=["Region", "Type", "Code"],
    how="left",
).drop(columns=["Type", "Code"])


### Immigrant and Localists

In [214]:
immigrant_score_map = {
    "Low Density Immigrants": 5,
    "Medium Density Immigrants": 20,
    "High Density Immigrants": 40,
}
immigrant_score = df_polygon["ward_immigrant"].map(immigrant_score_map).fillna(0)

location_score_map = {
    "Inner City": 5,
    "Town": 5,
    "Outer City": 5,
    "Suburb": 0,
    "Rural": -5,
}
location_score = df_polygon["Location"].map(location_score_map).fillna(0)

region_score = df_polygon["Region"].map({"Highland": 5}).fillna(0)

county_type_score_map = {
    "S": -10, "A1": -10, "B1": -10, "B2": -10, "UNI": -10,
    "E2": -10, "F1": -10, "F2": -10,
    "C1": -5,
    "TECH": 0, "GAME": 0, "C2": 0,
    "A2": 5, "E1": 5,
    "D1": 10, "D2": 10,
}
county_type_score = df_polygon["County_Type"].map(county_type_score_map).fillna(0)

has_immigrant_data = df_polygon["ward_immigrant"].notna()
full_score = df_polygon["D2"] + immigrant_score + location_score + region_score + county_type_score

df_polygon["D2_new_scores"] = (
    full_score.where(has_immigrant_data, df_polygon["D2"]).clip(lower=0, upper=80)
)


In [215]:
localist_score_map = {
    "Low Density Localists": 10,
    "Medium Density Localists": 20,
    "High Density Localists": 35,
}
localist_score = df_polygon["ward_localists"].map(localist_score_map).fillna(0)

location_score_map = {
    "Inner City": -7.5,
    "Town": 2.5,
    "Outer City": 0,
    "Suburb": 2.5,
    "Rural": 2.5,
}
location_score = df_polygon["Location"].map(location_score_map).fillna(0)

county_type_score_map = {
    "S": -10, "A1": -10, "B1": -10, "B2": -10, "UNI": -10,
    "E2": +10, "F1": +8, "F2": +8,
    "C1": -10,
    "TECH": 0, "GAME": 0, "C2": -10,
    "A2": 0, "E1": 0,
    "D1": 5, "D2": 5,
}
county_type_score = df_polygon["County_Type"].map(county_type_score_map).fillna(0)

has_localist_data = df_polygon["ward_localists"].notna()
full_score = df_polygon["F2"] + localist_score + location_score + county_type_score

df_polygon["F2_new_scores"] = (
    full_score.where(has_localist_data, df_polygon["F2"]).clip(lower=0, upper=85)
)


In [216]:
# avoid both exceeding 90
combined_score = df_polygon["D2_new_scores"] + df_polygon["F2_new_scores"]
needs_capping = combined_score >= 90

scale_factor = (90 / combined_score).where(needs_capping, 1.0)

df_polygon["D2_new_scores"] = df_polygon["D2_new_scores"] * scale_factor
df_polygon["F2_new_scores"] = df_polygon["F2_new_scores"] * scale_factor

df_polygon = df_polygon.drop(columns=["D2", "F2"])

cluster_cols_to_normalize = ["S", "A1", "A2", "B1", "B2", "C1", "C2", "D1", "E1", "E2", "F1"]

remaining_share = 100 - df_polygon["D2_new_scores"] - df_polygon["F2_new_scores"]
current_sum = df_polygon[cluster_cols_to_normalize].sum(axis=1)
normalize_factor = remaining_share / current_sum

df_polygon[cluster_cols_to_normalize] = df_polygon[cluster_cols_to_normalize].multiply(normalize_factor, axis=0)



In [217]:
# df_polygon["immigrants"] = df_polygon["D2_new_scores"] / 100 * df_polygon["Population"]
# df_polygon["localists"] = df_polygon["F2_new_scores"] / 100 * df_polygon["Population"]

# immigrants_total = df_polygon["immigrants"].sum()
# localists_total = df_polygon["localists"].sum()

# print("Total immigrants:", immigrants_total)
# print("Total localists:", localists_total)
# print("Combined total:", immigrants_total + localists_total)


### Population

In [218]:
sys.path.append(r"E:\Coding Site\just_for_fun")
from random_no_generation import random_normal_adv

df_polygon = df_polygon.rename(columns={"D2_new_scores": "D2", "F2_new_scores": "F2"})
check_cols = ["S", "A1", "A2", "B1", "B2", "C1", "C2", "D1", "E1", "E2", "F1", "D2", "F2"]

df_polygon[check_cols] = df_polygon[check_cols].div(100).multiply(df_polygon["Population"], axis=0)

means = df_polygon[check_cols].to_numpy().flatten().tolist()
sds = [mean * 0.1 for mean in means]

random_values = random_normal_adv(means, sds)
random_values = np.clip(random_values, 0, None).astype(int).reshape(len(df_polygon), len(check_cols))

row_sum = random_values.sum(axis=1)
adjustment = df_polygon["Population"].round().astype(int).to_numpy() - row_sum
max_col_idx = random_values.argmax(axis=1)
random_values[np.arange(len(random_values)), max_col_idx] += adjustment

df_polygon[check_cols] = random_values
df_polygon.columns


E:\Coding Site\just_for_fun\random_no_generation.py:27: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_FILE, header=None)


Index(['County', 'District', 'Shape_ID', 'geometry', 'Area', 'Region',
       'Location', 'County_Type', 'Population', 'District_Ward',
       'Constituency', 'centroid', 'ward_immigrant', 'ward_localists', 'S',
       'A1', 'A2', 'B1', 'B2', 'C1', 'C2', 'D1', 'E1', 'E2', 'F1', 'D2', 'F2'],
      dtype='object')

In [ ]:
# first save
df_population = df_polygon[['County', 'District', 'Shape_ID', 'S',
       'A1', 'A2', 'B1', 'B2', 'C1', 'C2', 'D1', 'E1', 'E2', 'F1', 'D2', 'F2']]
df_population.to_csv(r"E:\Coding Site\just_for_fun\Electoral_Modelling\csv_output\Population_Demographics.csv", index=False)

### Electorate

In [220]:
demographic_cols = ["S", "A1", "A2", "B1", "B2", "C1", "C2", "D1", "D2", "E1", "E2", "F1", "F2"]

constituency_lookup = df_polygon[["Shape_ID", "Constituency"]]
df_population = df_population.merge(constituency_lookup, on="Shape_ID", how="left")

sub_region_lookup = sub_region_map[["Constituency", "Sub_Region"]]
df_population = df_population.merge(sub_region_lookup, on="Constituency", how="left")

rate_lookup = data_electorate.set_index("Sub_Region")[demographic_cols].add_suffix("_rate")
df_population = df_population.merge(rate_lookup, left_on="Sub_Region", right_index=True, how="left")

means_df = (
    df_population[demographic_cols].to_numpy()
    * df_population[[c + "_rate" for c in demographic_cols]].to_numpy()
    / 100
)
means = means_df.flatten().tolist()
sds = [m * 0.1 for m in means]

random_values = random_normal_adv(means, sds)
random_values = np.clip(random_values, 0, None).astype(int).reshape(len(df_population), len(demographic_cols))

df_electorates = df_population[["County", "District", "Shape_ID", "Constituency", "Sub_Region"]].copy()
df_electorates[demographic_cols] = random_values

# electorate cannot exceed the eligible population in a given sector, and shouldn't
# fall too far below it either (elect/pop < 0.45): redraw offending cells instead
# of simply capping them to the population value
pop_values = df_population[demographic_cols].to_numpy()
elect_values = df_electorates[demographic_cols].to_numpy()
means_arr = np.array(means).reshape(elect_values.shape)
sds_arr = np.array(sds).reshape(elect_values.shape)

max_attempts = 10
for _ in range(max_attempts):
    with np.errstate(divide="ignore", invalid="ignore"):
        rate_values = np.where(pop_values > 0, elect_values / pop_values, np.nan)
    too_low = rate_values < 0.45
    too_high = rate_values > 0.975
    needs_redo = too_low | too_high
    if not needs_redo.any():
        break

    redo_values = random_normal_adv(
        means_arr[needs_redo].tolist(),
        sds_arr[needs_redo].tolist(),
    )
    elect_values[needs_redo] = np.clip(redo_values, 0, None).astype(int)

# fallback: cap any cells that still exceed population after max_attempts redraws
exceeds_population = elect_values > pop_values
elect_values[exceeds_population] = pop_values[exceeds_population]

df_electorates[demographic_cols] = elect_values

df_electorates


E:\Coding Site\just_for_fun\random_no_generation.py:27: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_FILE, header=None)
E:\Coding Site\just_for_fun\random_no_generation.py:27: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_FILE, header=None)
E:\Coding Site\just_for_fun\random_no_generation.py:27: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_FILE, header=None)
E:\Coding Site\just_for_fun\random_no_generation.py:27: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_FILE, header=None)
E:\Coding Site\just_for_fun\random_no_generation.py:27: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_FILE, header=None)
E:\Coding 

,County,District,Shape_ID,Constituency,Sub_Region,S,A1,A2,B1,B2,C1,C2,D1,D2,E1,E2,F1,F2
0,A01,Alington City,A01_001,The City of Alington,Capital East,0,78,1267,151,247,363,515,1488,317,529,2468,479,161
1,A01,Alington City,A01_002,The City of Alington,Capital East,308,2704,653,2004,139,237,257,568,145,60,241,0,0
2,A01,Alington City,A01_003,The City of Alington,Capital East,0,35,55,173,153,240,136,625,37,681,237,1398,177
3,A01,Alington City,A01_004,The City of Alington,Capital East,126,52,40,288,45,281,263,3619,189,422,116,96,66
4,A01,Alington City,A01_005,The City of Alington,Capital East,0,45,610,84,144,232,302,700,186,342,932,241,117
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20845,WA06,Aqua Corner,WA06_001,Upper Eastland and Readington,Southeast,0,0,0,7,6,15,13,43,0,34,12,140,1889
20846,WA07,Aqua Corner,WA07_001,Upper Eastland and Readington,Southeast,0,0,0,8,4,12,16,43,0,37,10,129,1951
20847,WA08,Aqua Corner,WA08_001,Upper Eastland and Readington,Southeast,0,0,0,6,5,12,12,43,0,40,13,156,1817
20848,WA09,Aqua Corner,WA09_001,Upper Eastland and Readington,Southeast,0,0,0,5,5,11,10,39,0,28,8,122,1535


In [223]:
bins = [0, 0.2, 0.45, 0.6, 0.8,0.9,0.999, 1.0,1.0+1e-9, np.inf]
labels = ["0-0.2", "0.2-0.45", "0.4-0.6", "0.6-0.8", "0.8-0.9","0.9-0.999","0.999-1.0", "1.0","1.0+"]

ratio_flat = ratio.to_numpy().flatten()
ratio_flat = ratio_flat[~np.isnan(ratio_flat)]

bucket_counts = pd.cut(ratio_flat, bins=bins, labels=labels, include_lowest=True).value_counts().sort_index()

print("bad cells (population == 0, electorate > 0):", bad_cells.values.sum())
print(bucket_counts)


bad cells (population == 0, electorate > 0): 0
0-0.2             2
0.2-0.45          3
0.4-0.6       15204
0.6-0.8      186372
0.8-0.9       42693
0.9-0.999      5606
0.999-1.0      1050
1.0               0
1.0+              0
Name: count, dtype: int64


In [224]:
df_electorates.to_csv(r"E:\Coding Site\just_for_fun\Electoral_Modelling\csv_output\Electorate_Demographics.csv", index=False)